## Импорт библиотек и загрузка данных

In [2]:
import os
print(os.getcwd())
print(os.listdir())

d:\Programs\analysis.ipynb
['.git', 'ad_spend.csv', 'edtech_data.db', 'final_work_database.db', 'funnel_events.csv', 'main.ipynb', 'payments.csv', 'README.md', 'test_results.csv', 'trial_lessons.csv', 'users.csv']


In [3]:
import pandas as pd
import sqlite3
import os



# Создаем подключение к базе
conn = sqlite3.connect('edtech_data.db')


pd.read_csv('users.csv').to_sql('users', conn, if_exists='replace', index=False)
pd.read_csv('payments.csv').to_sql('payments', conn, if_exists='replace', index=False)
pd.read_csv('trial_lessons.csv').to_sql('trial_lessons', conn, if_exists='replace', index=False)
pd.read_csv('ad_spend.csv').to_sql('ad_spend', conn, if_exists='replace', index=False)
pd.read_csv('funnel_events.csv').to_sql('funnel_events', conn, if_exists='replace', index=False)
pd.read_csv('test_results.csv').to_sql('test_results', conn, if_exists='replace', index=False)

print("Все таблицы загружены!")

Все таблицы загружены!


# Подготовка данных 

## Проверка количества строк в каждой таблице

Вывод: Данные успешно загружены. В таблице users 1500 пользователей, в funnel_events 5570 событий, в payments 163 платежа. Объёма данных достаточно для первичного анализа пользовательской воронки, каналов и оплат.

In [4]:
query = """SELECT COUNT(*) AS strings_users
,(SELECT COUNT(*) FROM funnel_events ) AS strings_funnel_events
,(SELECT COUNT(*) FROM test_results) AS strings_test_results
,(SELECT COUNT(*) FROM trial_lessons) AS strings_trial_lessons
,(SELECT COUNT(*) FROM payments) AS strings_payments
,(SELECT COUNT(*) FROM ad_spend ) AS strings_ad_spend
 FROM users """
pd.read_sql(query, conn)

,strings_users,strings_funnel_events,strings_test_results,strings_trial_lessons,strings_payments,strings_ad_spend
0,1500,5570,986,361,163,1620


## Уникальность user_id в таблице users

Вывод: Количество уникальных user_id совпадает с общим количеством строк в таблице users. Это значит, что в таблице нет дубликатов пользователей, и её можно использовать как базовую таблицу для анализа воронки и каналов.

In [5]:
query = """
SELECT 
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(*) AS total_rows
FROM users;

"""
pd.read_sql(query, conn)

,unique_users,total_rows
0,1500,1500


## Есть ли пользователи в событиях, которых нет в users

Вывод: Аномальных ID не обнаружено(Перед проблемных мест в пути пользователя по группам,нужно убедиться, что мы работаем с одинаковыми пользователями и данные собраны и интерпретированы правильно)

In [6]:
query = "SELECT DISTINCT user_id FROM  funnel_events WHERE user_id not in(SELECT DISTINCT user_id FROM users) "
pd.read_sql(query, conn)

,user_id


## Пропуски в ключевых полях

Вывод:Критичных пропусков в ключевых полях не обнаружено. Пропуски в test_completed_at, test_score, recommended_level и recommended_plan связаны с тем, что часть пользователей начала тест, но не завершила его. Пропуски в trial_attended_at и teacher_rating связаны с тем, что часть пользователей записалась на пробный урок, но не посетила его. Такие пропуски являются частью пользовательского поведения и могут быть использованы в дальнейшем анализе отвалов.

### Пропуски user_id

In [7]:
query = """SELECT COUNT(*) AS id_null_users
 ,(SELECT COUNT(*) FROM funnel_events WHERE user_id IS NULL ) AS id_null_funnel_events
 ,(SELECT COUNT(*) FROM payments WHERE user_id IS NULL) AS id_null_payments
 ,(SELECT COUNT(*) FROM trial_lessons WHERE user_id IS NULL) AS id_null_trial_lessons
 ,(SELECT COUNT(*) FROM test_results WHERE user_id IS NULL) AS id_null_test_results
 FROM users WHERE user_id IS NULL
 """
pd.read_sql(query, conn)

,id_null_users,id_null_funnel_events,id_null_payments,id_null_trial_lessons,id_null_test_results
0,0,0,0,0,0


### Пропуски event_name и event_time

In [8]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_name IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


In [9]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_time IS NULL"
pd.read_sql(query, conn) 

,COUNT(*)
0,0


### Пропуски acquisition_channel

In [10]:
query = """
SELECT COUNT(*) AS channel_null_funnel_events
,(SELECT COUNT(*) FROM payments WHERE acquisition_channel IS NULL) AS channel_null_payments
,(SELECT COUNT(*) FROM ad_spend WHERE acquisition_channel IS NULL) AS channel_null_ad_spend
,(SELECT COUNT(*) FROM users WHERE acquisition_channel IS NULL) AS channel_null_users
FROM funnel_events WHERE acquisition_channel IS NULL
"""
pd.read_sql(query, conn) 

,channel_null_funnel_events,channel_null_payments,channel_null_ad_spend,channel_null_users
0,0,0,0,0


### Пропуски payment_status and amount_eur 

In [11]:
query = "SELECT COUNT(*) FROM payments WHERE payment_status IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


In [12]:
query = "SELECT COUNT(*) FROM payments WHERE amount_eur  IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


### Провека всех строк

In [13]:
trial_lessons = pd.read_sql("SELECT * FROM trial_lessons", conn)
print(trial_lessons.isnull().sum())
print()
test_results = pd.read_sql("SELECT * FROM test_results", conn)
print(test_results.isnull().sum())
print()
users = pd.read_sql("SELECT * FROM users", conn)
print(users.isnull().sum())
print()
ad_spend = pd.read_sql("SELECT * FROM ad_spend", conn)
print(ad_spend.isnull().sum())
print()
funnel_events = pd.read_sql("SELECT * FROM funnel_events", conn)
print(funnel_events.isnull().sum())
print()
payments = pd.read_sql("SELECT * FROM payments", conn)
print(payments.isnull().sum())

user_id               0
trial_booked_at       0
trial_attended_at    79
trial_attended        0
teacher_rating       79
dtype: int64

user_id                0
test_started_at        0
test_completed_at    136
test_score           136
recommended_level    136
recommended_plan     136
dtype: int64

user_id                0
first_visit_at         0
acquisition_channel    0
campaign               0
device                 0
age_group              0
country                0
landing_page           0
user_goal              0
dtype: int64

date                   0
acquisition_channel    0
campaign               0
spend_eur              0
impressions            0
clicks                 0
dtype: int64

event_id               0
user_id                0
event_time             0
event_name             0
step_order             0
acquisition_channel    0
campaign               0
dtype: int64

payment_id             0
user_id                0
payment_date           0
plan_name              0
subscripti

## Дубликаты событий

Вывод:Дубликаты по event_id не обнаружены. Также не выявлены повторяющиеся события по комбинации user_id, event_time и event_name.

In [14]:
query = """SELECT 
    COUNT(DISTINCT event_id) AS unique_event_ids,
    COUNT(*) AS total_events
FROM funnel_events

 """
pd.read_sql(query, conn)

,unique_event_ids,total_events
0,5570,5570


In [15]:
query = """SELECT 
    user_id,
    event_time,
    event_name,
    COUNT(*) AS cnt
FROM funnel_events
GROUP BY 
    user_id,
    event_time,
    event_name
HAVING COUNT(*) > 1

 """
pd.read_sql(query, conn)

,user_id,event_time,event_name,cnt


## Корректность дат

Вывод: В таблице funnel_events не обнаружены случаи, когда события пользователя идут в обратном хронологическом порядке.

Не обнаружено случаев оплаты раньше первого визита

Не обнаружено посещений раньше записи

Даты коректны и готовы к работе

In [16]:
query = """ 
WITH tb AS( SELECT user_id
,event_time AS t1
,LAG(event_time,1) OVER(PARTITION BY user_id ORDER BY event_time) AS t2
FROM funnel_events  )
SELECT user_id, t1 AS event_time FROM tb
 WHERE     t1 < t2         
"""
pd.read_sql(query, conn)

,user_id,event_time


In [17]:
query = """ 
SELECT 
    p.user_id,
    u.first_visit_at,
    p.payment_date
FROM payments p
JOIN users u 
    ON p.user_id = u.user_id
WHERE p.payment_date < DATE(u.first_visit_at     )   
"""
pd.read_sql(query, conn)

,user_id,first_visit_at,payment_date


In [18]:
query = """ 
SELECT 
    user_id,
    trial_booked_at,
    trial_attended_at
FROM trial_lessons
WHERE trial_attended_at IS NOT NULL
  AND trial_attended_at < trial_booked_at

"""

pd.read_sql(query, conn)

,user_id,trial_booked_at,trial_attended_at


## Корректность Платежей

ВЫВОДЫ: 

Каждый payment_id встречается один раз, дубликатов платежей не обнаружено. По статусам видно, что часть платежей завершилась ошибкой, поэтому этот этап требует отдельного анализа.

В текущей таблице payments нет платежей со статусом refunded. Это не обязательно означает проблему с возвратами. Нужно уточнить, действительно ли возвратов не было за период анализа или данные по возвратам хранятся отдельно.

Доля неуспешных платежей составляет 22.7%. Это заметная доля, поэтому этап оплаты требует дополнительной проверки. Возможные причины: технические ошибки, неудобный платёжный сценарий, недостаток способов оплаты, ошибки со стороны пользователя или отсутствие повторной попытки после failed payment.

ГИПОТЕЗЫ:

Гипотеза 1. Пользователи не повторяют попытку оплаты после ошибки, поэтому часть потенциальной выручки теряется.

Гипотеза 2. На этапе оплаты может быть техническая или UX-проблема, но для подтверждения нужны дополнительные данные: payment_method, error_code, device, country и повторные попытки оплаты.

Гипотеза 3. Пользователям может не хватать удобных способов оплаты.

РЕКОМЕНДАЦИИ:

1.Проверить коректность работы возвратов

2.Проверить коректность работы терминала оплаты или сбора данных с него

3.В случае коректной работы всех систем, Для пользователей которые ушли с ошибкой, провести ретаргетинг с предложением о скидке + Email-рассылку


In [19]:
query = """
SELECT 
    COUNT(DISTINCT payment_id) AS unique_payment_ids,
    COUNT(*) AS total_payments
FROM payments
"""
pd.read_sql(query, conn)

,unique_payment_ids,total_payments
0,163,163


In [20]:
query = """
SELECT 
    payment_status,
    COUNT(*) AS payments_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payments), 1) AS share_percent
FROM payments
GROUP BY payment_status
ORDER BY payments_count DESC

"""
pd.read_sql(query, conn)

,payment_status,payments_count,share_percent
0,success,126,77.3
1,failed,37,22.7


In [21]:
query = """
SELECT user_id 
,COUNT(payment_id) AS cnt_status
 FROM payments GROUP BY user_id   HAVING COUNT(payment_id) > 1

"""
pd.read_sql(query, conn)

,user_id,cnt_status


# Базовый анализ пользователей

Выводы: 

* 1.Наибольшее количество пользователей приходит из TikTok(27.3%) и Instagram(21.7 %), которые суммарно обеспечивают почти половину всех новых регистраций

* 2.Анализ внутренних кампаний показывает разную степень однородности трафика. В Google Ads и YouTube наблюдается наиболее стабильное распределение (разброс до 0.7%), что говорит о предсказуемости этих инструментов. В то же время в TikTok и Instagram выделяются явные лидеры (кампании со скидками).

* 3.Наибольшее количество пользователей используют Телефоны(66.3%)

* 4.Основным рынком является Германия, откуда приходит подавляющее большинство пользователей(71.4%)

* 5.Ключевым стимулом для изучения языка является развитие карьеры (28.3%), что важно учитывать при подготовке рекламных офферов.(28.3 %) 

* 6.Основной точкой входа для новых пользователей является страница прохождения теста на уровень языка (41.5%)

## SQL Запросы

In [22]:
query = """
SELECT acquisition_channel	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_channel_percent
 FROM users GROUP BY acquisition_channel ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

,acquisition_channel,all_users,users_channel_percent
0,TikTok,410,27.3 %
1,Instagram,326,21.7 %
2,Google Ads,235,15.7 %
3,YouTube,223,14.9 %
4,Partner Referral,154,10.3 %
5,Telegram,152,10.1 %


In [23]:


query = """
SELECT 
    acquisition_channel
    ,campaign
    ,COUNT(user_id) AS all_users_campaign
    ,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1) || ' %' AS users_campaign_percent
FROM users
GROUP BY acquisition_channel, campaign
ORDER BY SUM(COUNT(user_id)) OVER(PARTITION BY acquisition_channel) DESC, all_users_campaign DESC
"""
pd.read_sql(query, conn)

,acquisition_channel,campaign,all_users_campaign,users_campaign_percent
0,TikTok,tt_discount_challenge,142,9.5 %
1,TikTok,tt_short_lessons,134,8.9 %
2,TikTok,tt_viral_test,134,8.9 %
3,Instagram,ig_story_discount,116,7.7 %
4,Instagram,ig_reels_beginner,111,7.4 %
5,Instagram,ig_lookalike_students,99,6.6 %
6,Google Ads,ga_search_english_course,95,6.3 %
7,Google Ads,ga_competitor,72,4.8 %
8,Google Ads,ga_brand,68,4.5 %
9,YouTube,yt_long_form,78,5.2 %


In [24]:
query = """
SELECT device
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_device_percent
 FROM users GROUP BY device ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

,device,all_users,users_device_percent
0,mobile,995,66.3 %
1,desktop,387,25.8 %
2,tablet,118,7.9 %


In [25]:
query = """
SELECT country	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_country_percent
 FROM users GROUP BY country ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

,country,all_users,users_country_percent
0,Germany,1071,71.4 %
1,Austria,129,8.6 %
2,Other EU,107,7.1 %
3,Switzerland,101,6.7 %
4,Poland,92,6.1 %


In [26]:
query = """
SELECT user_goal	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_goal_percent
 FROM users GROUP BY user_goal ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

,user_goal,all_users,users_goal_percent
0,career,425,28.3 %
1,travel,272,18.1 %
2,conversation,248,16.5 %
3,exam,241,16.1 %
4,relocation,177,11.8 %
5,school_university,137,9.1 %


In [27]:
query = """
SELECT landing_page		
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_landing_page_percent
 FROM users GROUP BY landing_page	 ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

,landing_page,all_users,users_landing_page_percent
0,level_test,623,41.5 %
1,pricing,261,17.4 %
2,trial_lesson,259,17.3 %
3,grammar_materials,193,12.9 %
4,speaking_club,164,10.9 %


# Общая пользовательская воронка и Анализ отвалов в воронке 

Вывод: 
* Максимальное число пользователей уходит на этапе начала теста (514 человек) и бронирования пробного занятия (392 человека). 

*  Критически низкая конверсия наблюдается при переходе к бронированию пробного занятия (отвал 52.1%) и на этапе оплаты обучения (отвал 55.3%). Более половины аудитории, дошедшей до этих шагов, покидает воронку

* Отвал после первого визита может  указывать на несоответствие рекламного оффера содержимому страницы, проблемы с UX/UI (сложная навигация, долгая загрузка) в ожидании пользователя,недостаточную ценность бесплатного теста для пользователя

* Отвал после регистрации может означать не подходящие варианты записи на пробный урок,не интересные форматы проведения,нарушение ожиданий

* Отвал на этапе оплаты может означать проблемы в качестве пробного урока,скрипте продаж,несовпадение с ценовыми ожиданиями,недостаток вариантов оплаты(подписка,рассрочка,единоразовый платёж)

In [30]:
query = """
WITH t1 AS (
    SELECT 
        event_name,
        COUNT(DISTINCT user_id) AS users_count,
        CASE 
            WHEN event_name = 'site_visit' THEN 1
            WHEN event_name = 'test_started' THEN 2
            WHEN event_name = 'test_completed' THEN 3
            WHEN event_name = 'registration' THEN 4
            WHEN event_name = 'trial_booked' THEN 5
            WHEN event_name = 'trial_attended' THEN 6
            WHEN event_name = 'plan_selected' THEN 7
            WHEN event_name = 'payment_success' THEN 8
        END AS step_number
    FROM funnel_events
    WHERE event_name IN ('site_visit', 'test_started', 'test_completed', 
                         'registration', 'trial_booked', 'trial_attended', 
                         'plan_selected', 'payment_success')
    GROUP BY event_name)

SELECT
    event_name
    ,users_count
    ,ROUND(users_count * 100.0 / (SELECT MAX(users_count) FROM t1), 1)|| ' %' AS total_conv
    ,ROUND(users_count * 100.0 / LAG(users_count) OVER (ORDER BY users_count DESC), 1) || ' %' AS step_conv
    , LAG(users_count) OVER (ORDER BY users_count DESC) - users_count   AS users_churn
    , ROUND((LAG(users_count) OVER (ORDER BY users_count DESC) - users_count) * 100.0 / 
      LAG(users_count) OVER (ORDER BY users_count DESC), 1) || ' %' AS step_churn_percent
FROM t1
GROUP BY event_name ORDER BY step_number
       """
pd.read_sql(query, conn)

,event_name,users_count,total_conv,step_conv,users_churn,step_churn_percent
0,site_visit,1500,100.0 %,NaN,NaN,NaN
1,test_started,986,65.7 %,65.7 %,514.0,34.3 %
2,test_completed,850,56.7 %,86.2 %,136.0,13.8 %
3,registration,753,50.2 %,88.6 %,97.0,11.4 %
4,trial_booked,361,24.1 %,47.9 %,392.0,52.1 %
5,trial_attended,282,18.8 %,88.7 %,36.0,11.3 %
6,plan_selected,318,21.2 %,88.1 %,43.0,11.9 %
7,payment_success,126,8.4 %,44.7 %,156.0,55.3 %
